<div dir="rtl">
<h1>Logits را از مسیر واقعی مدل بازسازی کنید</h1>
<p>درس 51 از 76 · یک جمله را تا امتیاز بعدی دنبال کنیم · <code dir="ltr">45-trace</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-07/chapter-02/45-trace.html">📖 بازگشت به همین درس</a></p>
<p>Embedding، موقعیت، بلوک‌ها و Head را با trace همان مدل تطبیق دهید.</p><p>پیش‌نیاز: تمام اجزای MiniGPT و تفاوت trace با یک محاسبهٔ جداگانه را بشناسید.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>آیا ورودی Attention بلوک اول همان Token Embedding خام است؟ کدام دو عملیات پیش از آن انجام می‌شوند؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
model = MiniGPT(ModelConfig(12,8,8,2,2,0.)).eval()
ids = torch.tensor([[1,2,3,4]])
reference_trace = {}
with torch.no_grad():
    reference_logits,_ = model(ids,trace=reference_trace)
print('trace fields:',list(reference_trace))

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع trace_logits(Model, ids, use_positions=True) دیکشنری با کلیدهای combined، block_outputs و Logits برگرداند. combined جمع ورودی پیش از Dropout، block_outputs فهرست خروجی همهٔ بلوک‌ها و Logits خروجی Head پس از final_norm باشد. از زیرلایه‌ها استفاده کنید، نه model(ids).</p>
</div>

In [ ]:
def trace_logits(model, ids, use_positions=True):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = trace_logits(model,ids)
    if result is None: return False
    for tokens in (ids,torch.tensor([[4,3,2],[2,1,2]])):
        for positions in (False,True):
            actual = trace_logits(model,tokens,use_positions=positions)
            trace = {}; logits,_ = model(tokens,use_positions=positions,trace=trace)
            torch.testing.assert_close(actual['combined'],trace['combined_embedding'])
            assert len(actual['block_outputs']) == len(model.blocks)
            for a,b in zip(actual['block_outputs'],trace['layers']):
                torch.testing.assert_close(a,b['output'])
            torch.testing.assert_close(actual['logits'],logits)
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط use_positions را روی همان مدل خاموش کنید. شکل‌ها ثابت می‌مانند، ولی مسیر ورودی تغییر می‌کند. این حذف کنترل‌شدهٔ موقعیت به‌تنهایی دربارهٔ کیفیت زبان یا بی‌اهمیت‌بودن ترتیب حکم نمی‌دهد.</p>
</div>

In [ ]:
with torch.no_grad():
    with_positions = model(ids,use_positions=True)[0]
    without_positions = model(ids,use_positions=False)[0]
print('same shape:',with_positions.shape == without_positions.shape)
print('position ablation change:',(with_positions-without_positions).abs().max().item())

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>نسخهٔ خراب Attention را روی Embedding خام اجرا می‌کند. تابع first_attention_input(Model,ids) فقط ورودی صحیحِ Attention بلوک اول را برگرداند: جمع موقعیت، Dropout ورودی و norm_1 را فراموش نکنید.</p>
</div>

In [ ]:
with torch.no_grad():
    wrong_input = model.token_embedding(ids)
    correct_input = model.blocks[0].norm_1(reference_trace['layers'][0]['input'])
print('raw embedding/input difference:',(wrong_input-correct_input).abs().max().item())

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def first_attention_input(model, ids):
    # TODO
    return None

In [ ]:
def test_repair():
    result = first_attention_input(model,ids)
    if result is None: return False
    torch.testing.assert_close(result,correct_input)
    for tokens in (torch.tensor([[2]]),torch.tensor([[2,3],[4,5]])):
        trace = {}; model(tokens,trace=trace)
        torch.testing.assert_close(first_attention_input(model,tokens),model.blocks[0].norm_1(trace['layers'][0]['input']))
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>trace از forward واقعی MiniGPT می‌آید و مسیر دستی با تک‌تک خروجی‌های بلوک تطبیق داده شد. مدل eval و Dropout صفر است تا اختلاف تصادف را با اختلاف معماری اشتباه نگیریم.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>اگر فقط Logits نهایی متفاوت بود، کدام مقایسهٔ میانی به شما کمک می‌کرد اولین نقطهٔ انحراف را پیدا کنید؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-07/chapter-02/45-trace.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/45-trace.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>